# **Tracking Capacity Unit (CU) Consumption by User-Generated DAX Queries**

# Check out this [blog post](https://bits2bi.com/2025/10/05/tracking-capacity-unit-consumption-for-dax-queries-with-fabric-notebooks/) for details about this notebook

### Prepare Environment

In [ ]:
# Import the Semantic Link Python library
import sempy.fabric as fabric

# Import other libraries and modules
from datetime import datetime, timedelta
import time

import pandas as pd
import re

### Specify Targets Details

In [ ]:
# Declare and assign variables

## ........ PROVIDE THE NAME OR ID OF YOUR TARGET WORKSPACE, TARGET SEMANTIC MODEL, THE CAPACITY ID OF THE TARGET WORKSPACE IN FOLLOWING VARIABLES ........ ##

target_workspace = "<TARGET_WORKSPACE_NAME_OR_ID>"
target_semantic_model = "<TARGET_SEMANTIC_MODEL_NAME_OR_ID>"
target_capacity_Id = "<CAPACITY_ID_OF_TARGET_WORKSPACE>"

target_workspace = "1879c29f-aac0-4675-a2e8-7692826250a6"
target_semantic_model = "82a38141-0f3e-41ab-b1f8-36aafa192711"
target_capacity_Id = "d16efc02-0391-43bd-8841-47a044dd67ce"

### Specify Capacity Metrics Details

In [ ]:
## ........ PROVIDE THE NAME OR ID OF YOUR FABRIC CAPACTIY METRICS WORKSPACE AND FABRIC CAPACTIY METRICS SEMANTIC MODEL ........ ##

capacity_metric_workspace = "<CAPACITY_METRIC_WORKSPACE_NAME_OR_ID>"
capacity_metric_semantic_model = "<CAPACITY_METRIC_SEMANTIC_MODEL_NAME_OR_ID>"

capacity_metric_workspace = "a3f15aed-063c-4b37-b0c0-0387857c479b"
capacity_metric_semantic_model = "bec88303-d1b2-44c5-85fc-b5eede09cf4d"

# Tracing User-Generated DAX Queries

In [ ]:
# Define events to trace and their corresponding columns
# https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric.trace?view=semantic-link-python#sempy-fabric-trace-get-default-query-trace-schema

event_query_end_columns = [
  'EventClass',
  'EventSubclass',
  'RequestID',
  'ApplicationName',
  'NTCanonicalUserName',
  'SessionID',
  'ActivityID',
  'CurrentTime',
  'StartTime',
  'EndTime',
  'Duration',
  'CpuTime',
  'Success',
  'Error',
  'TextData'
]

# Define event schema with events and its associated columns
event_schema = {
    "QueryEnd": event_query_end_columns
}

#display(event_schema)

### Duration-Bound Trace
Run the trace for a predefined duration (e.g., 5 minutes). Once the time window expires, the trace stops automatically.

In [ ]:
## Duration-Based Trace

# Run the trace for a specified duration and collect the event at the end of the interval

# Set the total duration in seconds
total_duration = 60
check_interval = 20
elapsed_time = 0

# Establish a connection to the XMLA endpoint of the specific workspace and semantic model
# Using 'with' ensures the connection is closed automatically even if the code fails

with fabric.create_trace_connection(workspace = target_workspace, dataset = target_semantic_model) as trace_connection:
    # Define a trace session on the server side using the provided event_schema (e.g., Query End)
    with trace_connection.create_trace(event_schema, "SemPyDAXQueryTrace") as trace:

        # Start the trace. The argument (5) is a delay in seconds to wait for the trace to initialize
        trace.start(5)
        print("Trace started. Monitoring activity...")

        # Loop until the total duration is reached
        while elapsed_time < total_duration:
            time.sleep(check_interval)
            elapsed_time += check_interval
            print(f"Elapsed time: {elapsed_time} seconds...")

        ### CRITICAL: Final buffer to ensure the server flushes the last events
        print("Finalizing trace and flushing buffer...")
        time.sleep(5)

        # Stop the trace and download the data
        trace_logs = trace.stop()
        print("Trace complete")

display(trace_logs)

# Tracking Capacity Consumption

## Prepare Variables

#### Calculate TimePoint

In [ ]:
# The billing of Capacity Units starts from the beginning of the next TimePoint i.e., next 30 seconds mark.
# To automate the TimePoint in the DAX query to the Capacity Metric semantic model:
# 1. Get the current utc time in the dax_query_execution_time variable
# 2. Round up the time to the next 30 second mark
# 3. Format the time to fit the DAX query as DAX DATE() + TIME()
# This time will be used as the TimePoint in the DAX query to the Capacity Metric semantic model.

# 1. Get the maximum timestamp from the "Current Time" column
# We use pd.to_datetime just in case the column is currently a string
dax_query_execution_time = pd.to_datetime(trace_logs["Current Time"]).max()

# 2. Round up the time to the next 30 second mark
rounded_seconds = 30 if dax_query_execution_time.second < 30 else 0
extra_minutes = 1 if dax_query_execution_time.second >= 30 else 0

rounded_time = dax_query_execution_time.replace(
    second=rounded_seconds,
    microsecond=0
) + timedelta(minutes=extra_minutes)


# 3. Format the time to fit the DAX query as DAX DATE() + TIME()
operation_TimePoint = f"DATE({rounded_time.year}, {rounded_time.month}, {rounded_time.day}) + TIME({rounded_time.hour}, {rounded_time.minute}, {rounded_time.second})"

## ........ ALTERNATIVELY, SET THE TIMEPOINT MANUALLY ........ ##
# operation_TimePoint = "DATE(YYYY, MM, DD) + TIME(HH, MM, SS)"

print(operation_TimePoint)

#### Prepare the List of Operation IDs

In [ ]:
# "Request ID" from trace logs are referred to as "Operation Id" in Capacity Metrics
# Prepare a double quoted and comma separated list of Request IDs that will be used to track the Capacity consumption of each query
# Get unique, non-null Request IDs
unique_request_ids = trace_logs["Request ID"].dropna().unique()

# Map each ID to a double-quoted string and join them
request_id_list = ", ".join(map(lambda x: f'"{str(x).lower()}"', unique_request_ids))

print(request_id_list)

#### Set Wait Time

In [ ]:
# The details of an operation appears in the capacity metrics semantic model in approx 6 to 15 minutes
# The waitTime variable is used to set the sleep timer before running the DAX query to the Capacity Metric semantic model

## ........ INCREASE THE WAIT TIME IF YOU DON'T GET CAPACITY UNIT CONSUMPTION DETAILS ........ ##

# Set wait time in seconds
waitTime = 6 * 60  # 6 minutes

#### DAX Query to Track Capacity Consumption

In [ ]:
# DAX query to extract capacity consumption data from the capacity metrics semantic model

capacity_consumption_dax_query = f"""
// DAX query to return the capacity consumption by the interactive operation for a capacity during a TimePoint
DEFINE
    // Change the TimePoint MPARAMETER to get details of a desired TimePoint
    MPARAMETER 'TimePoint' =
        {operation_TimePoint}
    // Provide a capacity ID
    MPARAMETER 'CapacitiesList' = {{ "{target_capacity_Id}" }}
    VAR __OperationId =
        TREATAS ( {{ {request_id_list} }}, 'Timepoint Interactive Detail'[Operation Id] )
    VAR __TimePointInteractiveOperations =
        SUMMARIZECOLUMNS (
            'Timepoint Interactive Detail'[Operation Id],
            'Timepoint Interactive Detail'[Operation start time],
            'Timepoint Interactive Detail'[Operation end time],
            'Timepoint Interactive Detail'[Status],
            'Timepoint Interactive Detail'[Operation],
            'Timepoint Interactive Detail'[User],
            'Items'[Workspace Id],
            'Items'[Workspace name],
            'Items'[Item kind],
            'Items'[Item Id],
            'Items'[Item name],
            __OperationId,
            "Operation Duration (s)", CALCULATE ( SUM ( 'Timepoint Interactive Detail'[Duration (s)] ) ),
            "TimePoint CU", CALCULATE ( SUM ( 'Timepoint Interactive Detail'[Timepoint CU (s)] ) ),
            "Total CU", CALCULATE ( SUM ( 'Timepoint Interactive Detail'[Total CU (s)] ) )
        )

EVALUATE
__TimePointInteractiveOperations
ORDER BY
    'Timepoint Interactive Detail'[Operation start time] DESC,
    'Timepoint Interactive Detail'[Operation end time]
"""

#print(capacity_consumption_dax_query)

#### DAX Query to Hydrate the Capacity Metrics Semantic Model

In [ ]:
# DAX query to hydrate the capacity metrics semantic model

hydrate_the_model_dax_query = f"""
DEFINE
    MPARAMETER 'RegionName' = "Default"

EVALUATE
SUMMARIZECOLUMNS (
    'Capacities'[Capacity Id],
    'Capacities'[Capacity name],
    "Average_Utilization", 'All Measures'[Average utilization (last 1 hour)]
)
"""

## Execute DAX Query to Track Capacity Consumption

In [ ]:
# Wait at least 6 minutes before running the DAX query to the Capacity Metric semantic model to ensure operation details are available.

# If you have not opened your Capacity Metrics report in Power BI for some time, running the Capacity Unit extraction query may fail with an "Error obtaining data location" error.
# To fix this, run a "hydration" query to refresh your model so the data loads into memory and the extraction query works without errors.

# If the extraction query runs but returns no results, try increasing the waitTime to ensure the operation details are available in the model.

remaining_wait_time = 0

print(f"Waiting for {waitTime} seconds before extracting Capacity Unit consumption from the Capacity Metric semantic model to ensure operation details are available.")

while remaining_wait_time < waitTime:
    time.sleep(30)
    remaining_wait_time += 30
    print(f"Remaining wait time: {waitTime - remaining_wait_time} seconds...")

# Execute DAX query to hydrate the capacity metrics semantic model
print("Hydrating the capacity metrics semantic model!!!")

hydrate_the_model_result = fabric.evaluate_dax(
    workspace = capacity_metric_workspace, 
    dataset = capacity_metric_semantic_model,
    dax_string = hydrate_the_model_dax_query
)

# Execute DAX query to extract capacity consumption data from the capacity metrics semantic model
print("Getting CU consumption details!!!")

capacity_consumption_result = fabric.evaluate_dax(
    workspace = capacity_metric_workspace, 
    dataset = capacity_metric_semantic_model,
    dax_string = capacity_consumption_dax_query
)

display(capacity_consumption_result)

## Correlate Capacity Unit Consumption with the DAX Queries

In [ ]:
# Rename columns in capacity_consumption_result to only keep the text within []
# This function uses regex to find the content between brackets
def extract_bracket_content(col_name):
    match = re.search(r'\[(.*?)\]', col_name)
    return match.group(1) if match else col_name

capacity_consumption_result.columns = [extract_bracket_content(col) for col in capacity_consumption_result.columns]

# Join the two dataframes
# We create temporary normalized columns to perform the join
trace_logs['join_key'] = trace_logs['Request ID'].str.lower()
capacity_consumption_result['join_key'] = capacity_consumption_result['Operation Id'].str.lower()

# Perform the join
merged_df = pd.merge(
    trace_logs, 
    capacity_consumption_result, 
    on='join_key', 
    how='left'
)

# Show the requested columns
# We drop the temporary join_key and select the specific columns
capacity_consumption_by_query = merged_df[['Request ID', 'Duration', 'Cpu Time', 'Total CU', 'Operation Duration (s)', 'Text Data']]

display(capacity_consumption_by_query)